# Day 03: 3D Geometry, Quaternion Kinematics & Inertial Dead Reckoning
**State Estimation and Localization for Self-Driving Cars**

Today we transition from 2D planar motion to full 3D spatial orientation and inertial mechanics:
1. **3D Rotation Parameterizations**: Euler Angles (Gimbal Lock singularity) vs Direction Cosine Matrices (DCM) vs Hamiltonian Unit Quaternions.
2. **Quaternion Kinematics**: Angular velocity integration via the exponential map $\mathbf{q}_k = \mathbf{q}_{k-1} \otimes \operatorname{Exp}(\frac{1}{2} \boldsymbol{\omega} \Delta t)$.
3. **Strapdown IMU Dead Reckoning**: 3D numerical integration of specific force $\mathbf{f}_b$ and gravity compensation.
4. **IMU Error Budget**: Acceleration bias propagation leading to quadratic position drift ($e(t) \sim \frac{1}{2} \mathbf{b}_a t^2$).
5. **GNSS Pseudorange Trilateration**: Solving 3D receiver coordinates and clock bias via Gauss-Newton Least Squares.


In [ ]:
import numpy as np
import polars as pl
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from position_class import (
    Quaternion,
    StrapdownIMUIntegrator,
    solve_gnss_trilateration,
    skew_symmetric
)

print("Day 03 Modules Loaded Successfully!")


## 1. Quaternion Kinematics & Rotation Conversions

Unit quaternions $\mathbf{q} = [q_w, q_x, q_y, q_z]^T$ avoid the Euler angle gimbal lock singularity ($\theta = \pm 90^\circ$).


In [ ]:
# Quaternion multiplication & Euler angle verification
q1 = Quaternion.from_axis_angle(np.array([0, 0, 1]), np.radians(45.0))
q2 = Quaternion.from_axis_angle(np.array([0, 1, 0]), np.radians(30.0))
q_combined = q1.multiply(q2)

roll, pitch, yaw = q_combined.to_euler_angles()
print(f"Combined Euler Angles: Roll={np.degrees(roll):.2f} deg, Pitch={np.degrees(pitch):.2f} deg, Yaw={np.degrees(yaw):.2f} deg")
print(f"Rotation Matrix Norm Check (Orthonormality): {np.linalg.norm(q_combined.to_rotation_matrix() @ q_combined.to_rotation_matrix().T - np.eye(3)):.2e}")


## 2. Strapdown IMU Dead Reckoning & Quadratic Drift Divergence

We simulate 60 seconds of vehicle motion with a small constant accelerometer bias $b_{ax} = 0.05\text{ m/s}^2$.

Notice how unassisted IMU position diverges quadratically:

$$\Delta p(t) = \frac{1}{2} b_a t^2 \implies \Delta p(60\text{s}) = \frac{1}{2} (0.05) (3600) = 90.0\text{ meters!}$$


In [ ]:
# Simulate IMU Dead Reckoning with Bias
dt = 0.01  # 100 Hz IMU
total_time = 60.0
steps = int(total_time / dt)
t = np.linspace(0, total_time, steps)

# True motion: straight driving at 20 m/s with 9.81 m/s^2 gravity upward reaction
true_fb = np.array([0.0, 0.0, 9.81])
true_omega = np.array([0.0, 0.0, 0.0])

# Sensor corrupted with bias and white noise
accel_bias = np.array([0.05, 0.0, 0.0])  # 0.05 m/s^2 bias in X
noise_acc = 0.02

integrator_ideal = StrapdownIMUIntegrator(init_pos=np.zeros(3), init_vel=np.array([20.0, 0.0, 0.0]))
integrator_biased = StrapdownIMUIntegrator(init_pos=np.zeros(3), init_vel=np.array([20.0, 0.0, 0.0]))

pos_ideal, pos_biased = [], []

for k in range(steps):
    fb_ideal = true_fb
    fb_biased = true_fb + accel_bias + np.random.normal(0, noise_acc, 3)
    
    p_id, _, _ = integrator_ideal.step(fb_ideal, true_omega, dt)
    p_bi, _, _ = integrator_biased.step(fb_biased, true_omega, dt)
    
    pos_ideal.append(p_id)
    pos_biased.append(p_bi)

pos_ideal = np.array(pos_ideal)
pos_biased = np.array(pos_biased)
drift_error = pos_biased[:, 0] - pos_ideal[:, 0]

# Plot Drift Divergence
fig = make_subplots(rows=2, cols=1, subplot_titles=("<b>X-Position: Ideal vs Biased IMU Integration</b>", "<b>Quadratic Position Drift Error (m)</b>"))

fig.add_trace(go.Scatter(x=t, y=pos_ideal[:, 0], mode='lines', name='Ideal Trajectory', line=dict(color='green', width=2)), row=1, col=1)
fig.add_trace(go.Scatter(x=t, y=pos_biased[:, 0], mode='lines', name='Biased IMU (0.05 m/s²)', line=dict(color='red', width=2)), row=1, col=1)

fig.add_trace(go.Scatter(x=t, y=drift_error, mode='lines', name='Empirical Drift', line=dict(color='red', width=2)), row=2, col=1)
fig.add_trace(go.Scatter(x=t, y=0.5 * accel_bias[0] * (t**2), mode='lines', name='Theoretical Quadratic Curve (0.5 * b * t²)', line=dict(color='black', dash='dash')), row=2, col=1)

fig.update_layout(height=650, title_text="<b>IMU Dead Reckoning Drift Analysis</b>")
fig.show()


## 3. 3D GNSS Pseudorange Trilateration

Solving for vehicle 3D coordinate $[x, y, z]$ and clock offset $c \cdot \delta t$ from 6 satellite pseudorange measurements:

$$\rho_i = \|\mathbf{r}_{\text{sat}, i} - \mathbf{r}_{\text{rx}}\| + c \delta t_{\text{rx}} + \epsilon_i$$


In [ ]:
# True receiver position in ECEF frame (meters)
true_rx = np.array([1500000.0, -4500000.0, 4200000.0])
true_bias = 250.0  # meters

# 6 GNSS Satellite Constellation Positions
satellites = np.array([
    [15600000.0, -18000000.0, 20000000.0],
    [20000000.0, -12000000.0, 18000000.0],
    [10000000.0, -22000000.0, 16000000.0],
    [18000000.0, -16000000.0, 22000000.0],
    [12000000.0, -20000000.0, 24000000.0],
    [22000000.0, -14000000.0, 15000000.0]
])

# Generate noisy pseudoranges
ranges = np.linalg.norm(satellites - true_rx, axis=1) + true_bias + np.random.normal(0, 1.5, 6)

# Solve GNSS position via Gauss-Newton
est_pos, est_bias = solve_gnss_trilateration(satellites, ranges)

print(f"True Position:  {true_rx}")
print(f"Estimated Pos:  {est_pos}")
print(f"Position Error: {np.linalg.norm(est_pos - true_rx):.3f} meters")
print(f"Estimated Clock Bias: {est_bias:.2f} m (True: {true_bias:.2f} m)")
